# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


**Colours used in all charts:** every continent keeps the same colour in every figure, so the reader doesn't have to re-learn the colours from chart to chart. I used the Okabe–Ito palette from the lecture, which is colour-blind safe (no red/green pair). Grey always means context. The map in Q5 has its own diverging scale, because there the colour shows a number (% change), not a continent.

In [2]:
CONTINENT_COLOURS = {
    'Asia': '#0072B2',
    'Africa': '#E69F00',
    'Europe': '#56B4E9',
    'North America': '#009E73',
    'South America': '#CC79A7',
    'Oceania': '#000000',
}
GREY = '#BBBBBB'
LIGHT_GREY = '#DDDDDD'

---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:**

**Chart:** line chart. The question is about change over time, and a line shows how each continent moved from 1970 to 2022. Continents have very different sizes, so I divided each continent by its own 1970 population. Now every line starts at 1 and the y-axis simply means "how many times bigger than in 1970".

**Design choices:**
- Africa is the only coloured line (its colour from the palette), the other continents are grey for context.
- Lines are labelled directly at the end with their growth factor, so no legend is needed. Oceania, South America and Asia end almost at the same point, so they share one label.
- A light dotted line at 1 marks the 1970 level; no markers and no vertical gridlines.

**Answer:** Africa is growing by far the fastest: from 0.37 to 1.43 billion people, almost **4 times** its 1970 size. Oceania, South America and Asia a bit more than doubled (2.2–2.3x), North America grew 1.9x and Europe only 1.1x.

In [3]:
cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont.loc[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Growth'] = cont['Population'] / cont['Continent'].map(base)

end = cont.loc[cont['Year'] == 2022].set_index('Continent')['Growth'].sort_values()
fastest = end.index[-1]
cont['group'] = np.where(cont['Continent'] == fastest, fastest, 'Other')

fig = px.line(
    cont, x='Year', y='Growth',
    color='group', line_group='Continent',
    color_discrete_map={fastest: CONTINENT_COLOURS[fastest], 'Other': GREY},
    hover_name='Continent',
    hover_data={'group': False, 'Growth': ':.2f', 'Population': ':,'},
    labels={'Growth': 'Population relative to 1970 (1970 = 1)', 'Year': ''},
    title=f"{fastest}'s population grew {end[fastest]:.1f}x since 1970, the fastest of any continent, while Europe barely grew ({end['Europe']:.1f}x)",
    template='plotly_white',
    height=500, width=1000
)

fig.update_traces(line_width=1.5)
fig.update_traces(line_width=3.5, selector={'name': fastest})

fig.add_hline(y=1, line_dash='dot', line_color='#CCCCCC', line_width=1)

middle = ['Oceania', 'South America', 'Asia']
labels = [
    (f'<b>{fastest} {end[fastest]:.1f}x</b>', end[fastest], CONTINENT_COLOURS[fastest]),
    (f"North America {end['North America']:.1f}x", end['North America'], 'grey'),
    (f'Oceania, South America, Asia {end[middle].min():.1f}-{end[middle].max():.1f}x', end[middle].mean(), 'grey'),
    (f"Europe {end['Europe']:.1f}x", end['Europe'], 'grey'),
]
for text, y, colour in labels:
    fig.add_annotation(x=2022, y=y, text=text, showarrow=False, xanchor='left', xshift=8,
                       font=dict(color=colour, size=12))

fig.update_layout(
    showlegend=False,
    font=dict(family='Arial', size=13),
    margin=dict(r=260),
    yaxis=dict(gridcolor='#EEEEEE', zeroline=False, ticksuffix='x', range=[0.8, 4.1]),
    xaxis=dict(showgrid=False, tickvals=[1970, 1980, 1990, 2000, 2010, 2022])
)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:**

**Chart:** sorted horizontal bar chart. This is a top-15 ranking, and bars sorted by value make the order obvious. Horizontal bars keep the country names readable (no rotated text), and the bars start at zero, so the lengths are honest.

**Design choices:**
- African countries are orange (Africa's colour in all charts), all others grey, so you can count them at a glance. "Africa" in the title has the same colour, so no legend is needed.
- The value (in millions) is written at the end of each bar, so the x-axis is hidden.

**Answer:** India (+860M) and China (+603M) added by far the most people. **4 of the 15** countries are in Africa: Nigeria, Ethiopia, DR Congo and Egypt, together +413M people. Most of the list (8 countries) is in Asia. Together these 15 countries account for 2.8 of the 4.3 billion people added worldwide since 1970.

In [4]:
df['Added'] = df['2022 Population'] - df['1970 Population']

top15 = df.nlargest(15, 'Added').sort_values('Added').copy()
top15['Added_M'] = top15['Added'] / 1e6
top15['group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other')
n_africa = (top15['Continent'] == 'Africa').sum()
india_china = top15.loc[top15['Country'].isin(['India', 'China']), 'Added'].sum() / 1e9
orange = CONTINENT_COLOURS['Africa']

fig = px.bar(
    top15, x='Added_M', y='Country',
    orientation='h',
    color='group',
    color_discrete_map={'Africa': orange, 'Other': GREY},
    category_orders={'Country': top15['Country'].tolist()[::-1]},
    text='Added_M',
    hover_data={'group': False, 'Continent': True, 'Added_M': ':,.0f'},
    labels={'Added_M': 'People added 1970-2022 (millions)', 'Country': ''},
    title=(f'India and China alone added {india_china:.1f} billion people since 1970, and '
           f'<span style="color:{orange}"><b>{n_africa} of the top 15</b></span> are in '
           f'<span style="color:{orange}"><b>Africa</b></span>'),
    template='plotly_white',
    height=600, width=1000
)

fig.update_traces(texttemplate='%{text:,.0f}M', textposition='outside', cliponaxis=False)

fig.update_layout(
    showlegend=False,
    font=dict(family='Arial', size=13),
    xaxis=dict(visible=False),
    yaxis=dict(showgrid=False),
    bargap=0.25
)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:**

**Chart:** scatter plot. The question asks about the relationship between two numerical variables (land area and population) and about the points that don't follow it. Both variables go from tiny islands to huge countries, so I used a **log scale on both axes**; otherwise almost all countries would be squashed into one corner.

**Design choices:**
- Colour = continent, using the same continent colours as in the rest of the notebook. Here I kept a legend, because the question is about all six continents.
- A grey dashed trend line (fitted on the log values) is the reference: above the line means more people than expected for that area, below means fewer.
- Only the clear outliers are labelled, and the markers are slightly transparent because many points overlap.

**Answer:** Yes, bigger countries generally have more people (r = 0.87 on the log scale). But the continents differ: 78% of Asian countries lie above the trend (typically about 2x more people than expected for their size), while Oceania and South America are mostly below it (about half of what's expected). The clearest exceptions are Greenland and the Falkland Islands on the empty side, and Macau, Singapore and Bangladesh on the crowded side. This is only a correlation: area alone doesn't decide how many people live somewhere.

In [5]:
x_log = np.log10(df['Area'])
y_log = np.log10(df['2022 Population'])
slope, intercept = np.polyfit(x_log, y_log, 1)
r = np.corrcoef(x_log, y_log)[0, 1]

fig = px.scatter(
    df, x='Area', y='2022 Population',
    color='Continent',
    color_discrete_map=CONTINENT_COLOURS,
    category_orders={'Continent': list(CONTINENT_COLOURS)},
    hover_name='Country',
    hover_data={'Continent': False, 'Area': ':,.0f', '2022 Population': ':,.0f'},
    log_x=True, log_y=True,
    opacity=0.75,
    labels={'Area': 'Land area (km², log scale)', '2022 Population': 'Population 2022 (log scale)'},
    title=f'Bigger countries have more people (r = {r:.2f}), but most Asian countries sit above the trend and Greenland is the extreme exception',
    template='plotly_white',
    height=600, width=1100
)

fig.update_traces(marker=dict(size=9, line_width=0))

x_line = np.linspace(x_log.min(), x_log.max(), 50)
fig.add_scatter(x=10 ** x_line, y=10 ** (intercept + slope * x_line),
                mode='lines', line=dict(color='slategrey', dash='dash', width=1.5),
                hoverinfo='skip', showlegend=False)

outliers = {'Greenland': (40, -30), 'Falkland Islands': (40, 25), 'Macau': (40, -20),
            'Singapore': (40, 25), 'Bangladesh': (-60, -35), 'India': (-50, 10)}
for country, (ax, ay) in outliers.items():
    row = df.loc[df['Country'] == country].iloc[0]
    fig.add_annotation(x=np.log10(row['Area']), y=np.log10(row['2022 Population']),
                       text=country, showarrow=True, arrowhead=0, arrowcolor='grey',
                       ax=ax, ay=ay, font=dict(size=11, color='#333333'))

fig.add_annotation(x=np.log10(3), y=intercept + slope * np.log10(3),
                   text='trend line<br>above = more people than expected for the area',
                   showarrow=False, xanchor='left', yanchor='top', yshift=-10, align='left',
                   font=dict(size=11, color='slategrey'))

fig.update_layout(
    font=dict(family='Arial', size=13),
    legend=dict(title='', orientation='h', x=0, y=1.07),
    xaxis=dict(showgrid=False),
    yaxis=dict(gridcolor='#EEEEEE')
)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:**

**Chart:** box plot. It shows the median, the middle 50% and the outliers of several groups side by side, which is exactly "the spread, not just the average". A histogram works for one group only, and a violin plot would add shape detail that isn't reliable with only 14 countries in South America.

**Design choices:**
- Log scale, because density goes from 0.03 people/km² (Greenland) to over 23,000 (Macau); on a normal axis all boxes would be squashed to the left.
- Every country is shown as a dot on top of its box, so you can see how spread out each continent really is.
- The boxes are neutral grey, but every country dot has its continent's colour (the same as in the other charts), so this chart connects to the rest of the notebook. No continent is highlighted, because the question is about all of them.
- Boxes are sorted by median, the median values are written on the right, and the extreme countries are labelled.

**Answer:** Density differs much more **within** continents than **between** them. The medians go from 18 people/km² (South America) to 221 (North America), about 12x, but inside Asia alone it goes from 2 (Mongolia) to 23,172 (Macau). North America has the highest median only because 25 of its 40 territories are small, very dense Caribbean islands. Almost every continent has a few city-states or islands as extreme outliers, which is why the mean would be misleading here (e.g. Asia: mean about 1,025 vs median 123).

In [6]:
medians = df.groupby('Continent')['Density'].median().sort_values(ascending=False)
order = medians.index.tolist()
asia = df.loc[df['Continent'] == 'Asia']
asia_min = asia.loc[asia['Density'].idxmin()]
asia_max = asia.loc[asia['Density'].idxmax()]

fig = px.box(
    df, x='Density', y='Continent',
    color='Continent',
    color_discrete_map=CONTINENT_COLOURS,
    category_orders={'Continent': order},
    points='all',
    hover_name='Country',
    hover_data={'Continent': False, 'Density': ':,.1f'},
    log_x=True,
    labels={'Density': 'People per km² in 2022 (log scale)', 'Continent': ''},
    title=(f'Density varies far more within continents than between them: medians range from '
           f'{medians.min():.0f} to {medians.max():.0f} people/km²,<br>'
           f"but Asia alone goes from {asia_min['Density']:.0f} ({asia_min['Country']}) "
           f"to {asia_max['Density']:,.0f} ({asia_max['Country']})"),
    template='plotly_white',
    height=580, width=1050
)

fig.update_traces(
    fillcolor='rgba(0,0,0,0)', line=dict(color='#555555', width=1.5),
    marker=dict(size=6, opacity=0.6),
    pointpos=0, jitter=0.6, width=0.6
)

for country in ['Macau', 'Monaco', 'Greenland']:
    row = df.loc[df['Country'] == country].iloc[0]
    fig.add_annotation(x=np.log10(row['Density']), y=row['Continent'], text=country,
                       showarrow=False, yshift=18, font=dict(size=11, color='#333333'))

fig.add_annotation(xref='paper', x=1.02, y=order[0], yshift=32, text='<b>Median</b>',
                   showarrow=False, xanchor='left', font=dict(size=12, color='#333333'))
for continent, m in medians.items():
    fig.add_annotation(xref='paper', x=1.02, y=continent, text=f'{m:,.0f}/km²',
                       showarrow=False, xanchor='left', font=dict(size=12, color='#333333'))

fig.update_layout(
    showlegend=False,
    boxmode='overlay',
    font=dict(family='Arial', size=13),
    xaxis=dict(gridcolor='#EEEEEE', tickvals=[0.1, 1, 10, 100, 1000, 10000],
               ticktext=['0.1', '1', '10', '100', '1,000', '10,000']),
    yaxis=dict(showgrid=False),
    margin=dict(t=100, r=110)
)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:**

**Chart:** choropleth map. The question asks *where in the world* populations grew or shrank, so the geographic pattern itself is the insight. I used `px.choropleth` with the built-in ISO-3 shapes (`CCA3`), because this is a static overview, not a tool for zooming into one region.

**Design choices:**
- A **diverging scale** (`RdBu`) centred on 0, because % change can be positive or negative: blue = grew, red = shrank, white = no change. Red–blue is safe for colour-blind readers.
- The scale is capped at ±40%. A few countries grew extremely (Qatar +317%) and would wash out the rest of the map; the biggest decline is about -38%, so the shrinking countries still get their full colour. The cap is written on the figure and the real values are in the hover.
- "Shrank" and "grew" in the title use the two colours of the scale, so the title also works as the legend.

**Answer:** 34 countries shrank between 2000 and 2022. 18 of them are in Europe, mostly Eastern and Southern Europe (Lithuania, Latvia, Moldova, Bosnia and Herzegovina, Ukraine, Bulgaria...). The rest are mainly small islands in the Caribbean and the Pacific, plus Japan, Armenia and Georgia. The fastest growth was in Africa (median country +78%) and in the Gulf (Qatar +317%, UAE +188%). One limitation: very small island states are hard to see on a world map.

In [7]:
df['Change'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100
shrank = df.loc[df['Change'] < 0]
n_shrank = len(shrank)
n_europe = (shrank['Continent'] == 'Europe').sum()
red, blue = '#B2182B', '#2166AC'

fig = px.choropleth(
    df,
    locations='CCA3', locationmode='ISO-3',
    color='Change',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    range_color=[-40, 40],
    hover_name='Country',
    hover_data={'CCA3': False, 'Change': ':+.1f', '2000 Population': ':,', '2022 Population': ':,'},
    labels={'Change': 'Change 2000-2022 (%)'},
    title=(f'<span style="color:{red}"><b>{n_shrank} countries shrank</b></span> between 2000 and 2022, '
           f'{n_europe} of them in Europe, while Africa and the Gulf '
           f'<span style="color:{blue}"><b>grew</b></span> the fastest'),
    template='plotly_white',
    height=550, width=1100
)

fig.add_annotation(xref='paper', yref='paper', x=0, y=0, showarrow=False, xanchor='left',
                   text='Colour scale capped at ±40% (e.g. Qatar +317%); exact values on hover',
                   font=dict(size=10, color='slategrey'))

fig.update_layout(
    font=dict(family='Arial', size=13),
    geo=dict(showframe=False, showcoastlines=False, projection_type='natural earth'),
    coloraxis_colorbar=dict(title='Change<br>2000-2022', ticksuffix='%', thickness=15, len=0.7),
    margin=dict(l=0, r=0, t=60, b=20)
)
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:**

**Chart:** treemap. The data is a hierarchy (world → continent → country) and the question wants both levels in one figure. Rectangle area = 2022 population, which adds up correctly: countries add up to their continent and continents add up to the world. A stacked bar wouldn't work with 234 countries.

**Design choices:**
- Each continent has its own colour from the palette (the same as in the other charts), so the continents are easy to tell apart. Asia still stands out because of its size, and "Asia" in the title has the same blue. The parent boxes are light grey, so they don't compete with the countries.
- Every country box shows its share of the world, and each continent header shows the continent's total share, so both levels can be read directly without a legend.
- Tiny countries have no text, but their numbers are still in the hover.

**Answer:** almost 6 out of 10 people live in **Asia (59%)**, followed by Africa (18%), Europe (9%), North America (7.5%), South America (5.5%) and Oceania (0.6%). China and India alone hold 36% of the world population, which is 60% of Asia.

In [8]:
total = df['2022 Population'].sum()
cont_share = df.groupby('Continent')['2022 Population'].sum() / total * 100
china_india = df.loc[df['Country'].isin(['China', 'India']), '2022 Population'].sum() / total * 100
blue = CONTINENT_COLOURS['Asia']
tree = df.assign(Continent_label=df['Continent'].map(lambda c: f'{c} · {cont_share[c]:.1f}%'))

fig = px.treemap(
    tree,
    path=[px.Constant(f'World · {total / 1e9:.1f} billion people'), 'Continent_label', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_map=CONTINENT_COLOURS,
    title=(f'<span style="color:{blue}"><b>Asia</b></span> is home to {cont_share["Asia"]:.0f}% of the world\'s '
           f'{total / 1e9:.0f} billion people, and China and India alone hold {china_india:.0f}%'),
    template='plotly_white',
    height=720, width=1100
)

fig.update_traces(
    texttemplate='%{label}<br>%{percentRoot:.1%}',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} people<br>%{percentRoot:.1%} of the world<extra></extra>',
    marker_line_color='white', marker_line_width=1,
    root_color='white'
)

ids = fig.data[0].ids
fig.data[0].marker.colors = [
    c if ids[i].count('/') == 2 else '#E5E5E5'
    for i, c in enumerate(fig.data[0].marker.colors)
]

fig.update_layout(font=dict(family='Arial', size=13), margin=dict(l=10, r=10, t=60, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:**

**Chart:** 100% stacked bar chart, one bar per year. The question is about *shares* of a whole and how the mix changes, and there are only four groups, which is exactly the case where 100% stacked bars work well. Each bar is a snapshot of the world in that year. (The years in the data are not evenly spaced, so I show them as categories.)

**Design choices:**
- Africa is on the bottom (zero baseline) and Europe on the top (it ends at 100%, which is also a fixed baseline), so the two groups that change the most are the easiest to compare across years.
- Africa and Europe keep their colours from the other charts; Asia and the rest of the world are two greys, because they barely change.
- Percentage labels only on Africa and Europe, segment names written directly next to the last bar instead of a legend, and no y-axis since every bar is 100% anyway.

**Answer:** Africa's share of the world almost **doubled (10% → 18%)** and passed Europe's in the 1990s, while Europe's share **halved (18% → 9%)**. Asia's share hardly changed (58% → 59%, with a peak of 61% in 2000), and the rest of the world stayed at about 14%.

In [9]:
groups = {'Africa': 'Africa', 'Asia': 'Asia', 'Europe': 'Europe'}
share = df_long.assign(Group=df_long['Continent'].map(groups).fillna('Rest of the world'))
share = share.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
share['Share'] = share['Population'] / share.groupby('Year')['Population'].transform('sum')
share['Year'] = share['Year'].astype(str)
share['Label'] = np.where(share['Group'].isin(['Africa', 'Europe']), share['Share'].map('{:.0%}'.format), '')

s = share.pivot(index='Year', columns='Group', values='Share')
order = ['Africa', 'Asia', 'Rest of the world', 'Europe']
colours = {'Africa': CONTINENT_COLOURS['Africa'], 'Europe': CONTINENT_COLOURS['Europe'],
           'Asia': LIGHT_GREY, 'Rest of the world': GREY}

fig = px.bar(
    share, x='Year', y='Share',
    color='Group',
    color_discrete_map=colours,
    category_orders={'Group': order},
    text='Label',
    hover_data={'Label': False, 'Share': ':.1%'},
    labels={'Year': '', 'Group': ''},
    title=(f"<span style='color:{colours['Africa']}'><b>Africa's</b></span> share of the world almost doubled "
           f"({s.loc['1970', 'Africa']:.0%} → {s.loc['2022', 'Africa']:.0%}) and passed Europe's in the 1990s,<br>while "
           f"<span style='color:{colours['Europe']}'><b>Europe's</b></span> share halved "
           f"({s.loc['1970', 'Europe']:.0%} → {s.loc['2022', 'Europe']:.0%})"),
    template='plotly_white',
    height=580, width=1050
)

fig.update_traces(textposition='inside', textfont=dict(color='#222222', size=12),
                  marker_line_color='white', marker_line_width=1)

bottom = 0
for g in order:
    value = s.loc['2022', g]
    colour = colours[g] if g in ['Africa', 'Europe'] else 'grey'
    fig.add_annotation(x=len(s) - 1, y=bottom + value / 2, text=f'<b>{g}</b>' if colour != 'grey' else g,
                       showarrow=False, xanchor='left', xshift=45, font=dict(size=12, color=colour))
    bottom += value

fig.update_layout(
    showlegend=False,
    font=dict(family='Arial', size=13),
    bargap=0.3,
    yaxis=dict(showticklabels=False, showgrid=False, zeroline=False, title='', range=[0, 1]),
    xaxis=dict(showgrid=False, type='category'),
    margin=dict(r=160, t=100)
)
fig.show()